# Sign Language CNN - Improved Model
### Key improvements over original:
- Removed horizontal_flip (sign language is NOT mirror-symmetric)
- Added 3rd & 4th conv layers with increasing filters (32→64→128)
- Added BatchNormalization after each conv block
- Replaced Flatten+Dense with GlobalAveragePooling2D (reduces params, less overfitting)
- Added learning rate scheduler (ReduceLROnPlateau)
- Added EarlyStopping + ModelCheckpoint to save best model
- Added rotation & brightness augmentation for better real-world generalization
- Increased epochs to 20 (early stopping will find the best)

### Importing the Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.12.0


### Part 1 - Data Preprocessing

In [2]:
# FIXED: Removed horizontal_flip=True
# Sign language gestures are NOT mirror-symmetric — flipping can create wrong labels
# Added rotation and brightness augmentation for better real-world robustness
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    rotation_range=10,           # small rotation (hand angle variation)
    brightness_range=[0.8, 1.2], # lighting variation
    width_shift_range=0.1,       # slight position shift
    height_shift_range=0.1
    # horizontal_flip intentionally removed
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [3]:
# Update these paths to your dataset location
TRAIN_PATH = 'F:/Final Year Project/fy-pro/backend/dataSet/trainingData'
TEST_PATH  = 'F:/Final Year Project/fy-pro/backend/dataSet/testingData'

training_set = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=(128, 128),
    batch_size=32,             # increased from 10 for faster, more stable training
    color_mode='grayscale',
    class_mode='categorical'
)

test_set = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=(128, 128),
    batch_size=32,
    color_mode='grayscale',
    class_mode='categorical'
)

Found 12845 images belonging to 27 classes.
Found 4268 images belonging to 27 classes.


### Part 2 - Building the Improved CNN

In [4]:
classifier = tf.keras.models.Sequential([

    # Block 1: 32 filters
    tf.keras.layers.Conv2D(32, kernel_size=3, padding='same', activation='relu', input_shape=[128, 128, 1]),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(32, kernel_size=3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(pool_size=2, strides=2),
    tf.keras.layers.Dropout(0.25),

    # Block 2: 64 filters — learns more complex features
    tf.keras.layers.Conv2D(64, kernel_size=3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(64, kernel_size=3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(pool_size=2, strides=2),
    tf.keras.layers.Dropout(0.25),

    # Block 3: 128 filters — high-level features
    tf.keras.layers.Conv2D(128, kernel_size=3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(128, kernel_size=3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPool2D(pool_size=2, strides=2),
    tf.keras.layers.Dropout(0.25),

    # GlobalAveragePooling replaces Flatten — reduces 4M+ params to ~16K
    tf.keras.layers.GlobalAveragePooling2D(),

    # Classifier head
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.50),
    tf.keras.layers.Dense(27, activation='softmax')
])

classifier.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 128, 128, 32)      320       
                                                                 
 batch_normalization (BatchN  (None, 128, 128, 32)     128       
 ormalization)                                                   
                                                                 
 conv2d_1 (Conv2D)           (None, 128, 128, 32)      9248      
                                                                 
 batch_normalization_1 (Batc  (None, 128, 128, 32)     128       
 hNormalization)                                                 
                                                                 
 max_pooling2d (MaxPooling2D  (None, 64, 64, 32)       0         
 )                                                               
                                                        

### Part 3 - Training with Callbacks

In [5]:
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    # Save the best model (by val_accuracy), not just the last epoch
    ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # Stop early if val_accuracy stops improving for 5 epochs
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # Halve the learning rate if val_loss stagnates for 3 epochs
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

In [6]:
history = classifier.fit(
    training_set,
    epochs=20,
    validation_data=test_set,
    callbacks=callbacks
)

Epoch 1/20
402/402 [==============================] - ETA: 0s - loss: 1.3636 - accuracy: 0.5927
Epoch 1: val_accuracy improved from -inf to 0.03608, saving model to best_model.h5
402/402 [==============================] - 833s 2s/step - loss: 1.3636 - accuracy: 0.5927 - val_loss: 9.9785 - val_accuracy: 0.0361 - lr: 0.0010
Epoch 2/20
402/402 [==============================] - ETA: 0s - loss: 0.1910 - accuracy: 0.9506
Epoch 2: val_accuracy improved from 0.03608 to 0.76593, saving model to best_model.h5
402/402 [==============================] - 796s 2s/step - loss: 0.1910 - accuracy: 0.9506 - val_loss: 0.9904 - val_accuracy: 0.7659 - lr: 0.0010
Epoch 3/20
402/402 [==============================] - ETA: 0s - loss: 0.0672 - accuracy: 0.9851
Epoch 3: val_accuracy did not improve from 0.76593
402/402 [==============================] - 725s 2s/step - loss: 0.0672 - accuracy: 0.9851 - val_loss: 1.4613 - val_accuracy: 0.6403 - lr: 0.0010
Epoch 4/20
402/402 [==============================] - ETA

### Part 4 - Visualize Training

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show() 

### Part 5 - Save the Model

In [11]:
model_json = classifier.to_json()
with open('model_new.json', 'w') as json_file:
    json_file.write(model_json)
print('Model architecture saved')

classifier.save_weights('model_new.h5')
print('Weights saved')

# Also save as full SavedModel format (recommended for modern TF)
classifier.save('sign_language_model_full.h5')
print('Full model saved')

Model architecture saved
Weights saved
Full model saved
